# Домашнее задание № 5 

В данном задании требуется реализовать некоторые из метрик, рассмотренные на лекции.

Все функции, кроме ```compute_gain```, в качестве первых двух аргументов принимают на вход тензоры ```ys_true``` и ```ys_pred```. Это вещественные тензоры ```pytorch``` размерности ```n```, задающие целевые отметки релевантности и предсказанные значения соответственно. 

Для генерации примеров входных данных можете использовать ```torch.rand(n)```, если не указана специфика исходных тензоров. 

Считается, что ```ys_pred``` содержит уникальные значения без повторений.

In [1]:
import math
from math import log2

import torch
from torch import Tensor, sort

## Swapped Pairs

```num_swapped_pairs``` — функция для расчёта количества неправильно упорядоченных пар (корректное упорядочивание — от наибольшего значения в ```ys_true``` к наименьшему) или переставленных пар.

In [2]:
def num_swapped_pairs(ys_true: Tensor, ys_pred: Tensor) -> int:
    ys_pred_si = ys_pred.sort(descending=True)[1]
    ys_true_s = torch.take(ys_true, ys_pred_si)
    num = 0
    for i in range(ys_true_s.shape[0]):
        for j in range(i):
            num += ys_true_s[i]>ys_true_s[j]
    return num

In [3]:
# не изменять
ys_true = torch.tensor([2, 1, 0, 1, 2])
ys_pred = torch.tensor([0.1, 0.3, 0.2, 0.13, 0.12])

res = num_swapped_pairs(ys_true, ys_pred)
print(res)  # 7

tensor(7)


## Gain

```compute_gain``` — вспомогательная функция для расчёта DCG и NDCG, рассчитывающая показатель Gain. Принимает на вход дополнительный аргумент — указание схемы начисления Gain (```gain_scheme```).

Необходимо реализовать метод при:
- ```gain_scheme="const"``` - постоянное начисление Gain
- ```gain_scheme="exp2"``` - рассчитываемый по формуле $(2^r −1)$, где $r$ — реальная релевантность документа некоторому запросу.

In [4]:
def compute_gain(y_value: float, gain_scheme: str) -> float:
    if( gain_scheme == "exp2"):
        return 2**y_value-1
    if( gain_scheme == "const"):
        return y_value

In [5]:
# не изменять
value = 5

res = compute_gain(value, 'exp2')
print(res)  # 31

res = compute_gain(value, 'const')
print(res)  # 5

31
5


## DCG

```dcg``` и ```ndcg``` — функции расчёта DCG и NDCG. Принимают на вход дополнительный параметр ```gain_scheme```, аналогичный таковому в функции ```compute_gain```.

In [6]:
def dcg(ys_true: Tensor, ys_pred: Tensor, gain_scheme: str) -> float:
    if( gain_scheme == "exp2"):
        ys_pred_si = ys_pred.sort(descending=True)[1]
        ys_true = torch.take(ys_true, ys_pred_si)
        
        gain = torch.pow(2, ys_true) - 1

        return torch.sum(gain / torch.log2(torch.arange(2, len(ys_true) + 2)))


def ndcg(ys_true: Tensor, ys_pred: Tensor, gain_scheme: str = 'const') -> float:
    if( gain_scheme == "exp2"):
        dcg_v = dcg(ys_true, ys_pred, gain_scheme)
        idcg_v = dcg(ys_true.sort(descending=True)[0], ys_pred.sort(descending=True)[0], gain_scheme)
        return dcg_v/idcg_v

In [7]:
# не изменять
ys_true = torch.tensor([2, 2, 4, 1, 2, 0])
ys_pred = torch.tensor([0.1, 0.3, 0.2, 0.14, 0.12, 0.6])

res = dcg(ys_true, ys_pred, gain_scheme='exp2')
res_n = ndcg(ys_true, ys_pred, gain_scheme='exp2')

print(f'{res:.15f}')  # 12.052645801815459
print(f'{res_n:.15f}')  # 0.6004804162123548

12.052645683288574
0.600480377674103


## Precission@k


```precission_at_k``` — функция расчёта точности в топ-k позиций для бинарной разметки (в ```ys_true``` содержатся только нули и единицы). Если среди лейблов нет ни одного релевантного документа (единицы), то необходимо вернуть -1. 

Функция принимает на вход параметр k, указывающий на то, по какому количеству объектов необходимо произвести расчёт метрики. Учтите, что k может быть больше количества элементов во входных тензорах.

In [8]:
def precission_at_k(ys_true: Tensor, ys_pred: Tensor, k: int) -> float:
    if (k==0):
        return 0
    ys_pred_si = ys_pred.sort(descending=True)[1]
    ys_true = torch.take(ys_true, ys_pred_si[:k])

    return sum(ys_true)/k

In [9]:
# не изменять
ys_true = torch.tensor([0, 0, 1, 1, 0, 1])
ys_pred = torch.tensor([0.1, 0.3, 0.2, 0.14, 0.12, 0.6])

res = precission_at_k(ys_true, ys_pred, k=3)
print(f'{res:.16f}')  # 0.6666666865348816

0.6666666865348816


## Average Precision

```average_precision``` — функция расчёта AP для бинарной разметки (в ```ys_true``` содержатся только нули и единицы). Если среди лейблов нет ни одного релевантного документа (единицы), то необходимо вернуть -1.

In [10]:
def average_precision(ys_true: Tensor, ys_pred: Tensor) -> float:
    p = 0
    for k in (ys_true==1).nonzero(as_tuple=True)[0]:
        p += precission_at_k(ys_true, ys_pred, k+1)
    
    return p/sum(ys_true)

In [11]:
# не изменять
ys_true = torch.tensor([1, 0, 1, 1, 0, 1, 0, 0])
ys_pred = torch.tensor([0.9, 0.8, 0.7, 0.6, 0.4, 0.3, 0.2, 0.1])

res = average_precision(ys_true, ys_pred)
print(f'{res:.16f}')  # 0.7708333333333333

0.7708333730697632


## reciprocal_rank

```reciprocal_rank``` — функция для расчёта MRR (без усреднения). В ```ys_true``` могут содержаться только нули и максимум одна единица. 

In [12]:
def reciprocal_rank(ys_true: Tensor, ys_pred: Tensor) -> float:
    ys_pred_si = ys_pred.sort(descending=True)[1]
    ys_true = torch.take(ys_true, ys_pred_si)
    if (not (ys_true==1).nonzero().numel()):
        return 0
    return 1/((ys_true==1).nonzero().item()+1)

In [13]:
# не изменять
ys_true = torch.tensor([0, 0, 0, 1, 0, 0])
ys_pred = torch.tensor([0.1, 0.3, 0.2, 0.14, 0.12, 0.6])

res = reciprocal_rank(ys_true, ys_pred)
print(res)  # 0.25

0.25


## p_found

```p_found``` — функция расчёта P-found от Яндекса, принимающая на вход дополнительный параметр ```p_break``` — вероятность прекращения просмотра списка документов в выдаче. Базовая вероятность просмотреть первый документ в выдаче ($pLook[0]$) равняется единице. ```ys_true``` нормированы от 0 до 1 (вероятность удовлетворения запроса пользователя).

In [35]:
def pRel(ys_true: Tensor, ys_pred: Tensor, k) -> float:
    ys_pred_si = ys_pred.sort(descending=True)[1]
    ys_true = torch.take(ys_true, ys_pred_si)
    return ys_true[k].item()

In [36]:
def pLook(ys_true: Tensor, ys_pred: Tensor, k, p_break: float = 0.15) -> float:
    if k == 0:
        return 1.0
    return pLook(ys_true, ys_pred,k - 1, p_break)*(1 - p_break)*(1 - pRel(ys_true, ys_pred,k - 1))

In [39]:
def p_found(ys_true: Tensor, ys_pred: Tensor, p_break: float = 0.15) -> float:
    p=0
    for i in range(ys_true.shape[0]):
        p += pLook(ys_true, ys_pred, i, p_break)*pRel(ys_true, ys_pred, i)
    return p

In [40]:
# не изменять
ys_true = torch.tensor([0, 0, 0, 1, 0, 1])
ys_pred = torch.tensor([0.91, 0.72, 0.12, 0.24, 0.15, 0.6])

res = p_found(ys_true, ys_pred, 0.12)
print(f'{res:.4f}')  # 0.7744

0.7744
